<a href="https://colab.research.google.com/github/parshav42/50_ML_models/blob/main/24modelCIFAR_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🧠 Concept First

CIFAR-10 vs MNIST/FashionMNIST:

	MNIST	FashionMNIST	CIFAR-10
Image size	28×28	28×28	32×32
Colors	Grayscale	Grayscale	RGB (3 channels)
Classes	10 digits	10 clothes	10 objects
Difficulty	Easy	Medium	Hard

CIFAR-10 classes:
Airplane, Car, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck

📥 Download CIFAR-10

Built into PyTorch — no Kaggle needed!

Use torchvision.datasets.CIFAR10
Apply transforms → ToTensor() and Normalize
Normalize with mean=(0.5, 0.5, 0.5) std=(0.5, 0.5, 0.5)
Batch size → 64
📌 Architecture To Build
Input → (3, 32, 32) RGB image
    ↓
Conv2d(3, 32, kernel_size=3, padding=1) + ReLU
MaxPool2d(2)
    ↓
Conv2d(32, 64, kernel_size=3, padding=1) + ReLU
MaxPool2d(2)
    ↓
Conv2d(64, 128, kernel_size=3, padding=1) + ReLU
MaxPool2d(2)
    ↓
Flatten()
Linear(128 * 4 * 4, 256) + ReLU
Dropout(0.5)
Linear(256, 10)
📌 Training Setup
Loss → nn.CrossEntropyLoss()
Optimizer → Adam lr=0.001
Epochs → 10
Print accuracy after each epoch
📌 Must Do

Part 1 — Show sample images:

Display 8 CIFAR images in a grid
Print their class names
See how hard they look!

Part 2 — Per epoch accuracy:

Train accuracy each epoch
Test accuracy each epoch
Plot both on same graph

Part 3 — Final results:

Which class is hardest to classify?
Print per class accuracy
🧠 Understand This
Why padding=1 in Conv2d?
Why 3 input channels instead of 1?
Why CIFAR harder than FashionMNIST?
What does Normalize do to images?
📊 Image Size Journey
Layer	Output Size
Input	3 × 32 × 32
Conv1 + Pool	32 × 16 × 16
Conv2 + Pool	64 × 8 × 8
Conv3 + Pool	128 × 4 × 4
Flatten	2048
Linear 1	256
Linear 2	10
🎯 Target
Accuracy → 70%+ after 10 epochs
CIFAR is hard — 70% is good!
Plot train vs test accuracy curve

In [ ]:
import torch
from torchvision import datasets
from torch.utils.data import DataLoader
from torchvision import transforms

In [ ]:
tr = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean = (0.5,0.5,0.5),
        std = (0.5,0.5,0.5)
    )
])

te = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean = (0.5,0.5,0.5),
        std = (0.5,0.5,0.5)
    )
])

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
train_t = datasets.CIFAR10(
    root = '/content/Untitled Folder',
    download = True,
    train = True,
    transform = tr,
    target_transform = None


)

test_t = datasets.CIFAR10(
    root='/content/Untitled Folder',
    download=True,
    train = False,
    transform = te,

)

In [ ]:
import matplotlib.pyplot as plt

img , lab = train_t[1]

# Matplotlib expects image data in (H, W, C) format, but PyTorch datasets often provide (C, H, W).
# We need to transpose the image dimensions.
# Also, the image was normalized, so we need to denormalize it for proper display.
img_display = img.permute(1, 2, 0) # Change from (C, H, W) to (H, W, C)
img_display = img_display * 0.5 + 0.5 # Denormalize from [-1, 1] to [0, 1]

plt.imshow(img_display)

In [ ]:
cass = train_t.classes
cass

In [ ]:
# for i in range(1,14):

#     img.la =

In [ ]:
from torch.utils.data import DataLoader


train_datal = DataLoader(
    train_t,
    batch_size =32,
    shuffle = True

    )

test_data = DataLoader(
    test_t,
    batch_size=32,

)

In [ ]:
from torch import nn


class tp(nn.Module):
    def __init__(self,input):
        super().__init__()

        self.con1 = nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.con2 = nn.Sequential(
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.con3 = nn.Sequential(
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.con4= nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 *4 * 4,256),
            nn.ReLU(),
            nn.Linear(256,10)

        )
    def forward(self,x):
        return self.con4(self.con3(self.con2(self.con1(x))))

In [ ]:
model = tp(1)
model = model.to(device)

loss_fn = nn.CrossEntropyLoss()

optim = torch.optim.Adam(model.parameters(),lr=0.001)


In [ ]:
epos =10

for epos in range(epos):

    for batch,(X,y) in enumerate(train_datal):

        X = X.to(device)
        y = y.to(device)
        model.train()

        y_pre = model(X)


        loss = loss_fn(y_pre,y)

        optim.zero_grad()

        loss.backward()

        optim.step()
    print(f'epos{epos+1}')


In [ ]:

model.eval()
test_loss, correct = 0, 0

with torch.no_grad():
    for X, y in test_data:
        # Forward pass
        X = X.to(device)
        y = y.to(device)
        pred = model(X)
        test_loss += loss_fn(pred, y).item()
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()

test_loss /= len(test_data)
correct /= len(test_data.dataset)

print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")